# Colab GPU Run: Experiment 02 — Noise Sensitivity

This notebook runs `experiments/exp02_noisy_data.py` on Google Colab with GPU acceleration.

1. In Colab, set **Runtime -> Change runtime type -> Hardware accelerator = GPU**.
2. Set `REPO_URL` to your GitHub repository URL.
3. Run cells in order.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/thlgrs/Vibration_PINN.git"  # <- update this
PROJECT_DIR = Path("/content/Vibration_PINN")

if not PROJECT_DIR.exists():
    if "YOUR-ORG" in REPO_URL:
        raise ValueError("Set REPO_URL to your real GitHub repo first.")
    subprocess.check_call(["git", "clone", REPO_URL, str(PROJECT_DIR)])
else:
    # Pull latest changes if repo already exists (avoids running stale code)
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"])

os.chdir(PROJECT_DIR)
print(f"Working directory: {Path.cwd()}")
print(f"Commit: {subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip()}")

In [ ]:
!nvidia-smi

In [ ]:
!python -m pip install --quiet --upgrade pip
!python -m pip install --quiet -r requirements-colab.txt

In [ ]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Quick Smoke Run

Runs 2 trials per noise level with few epochs to verify the pipeline.


In [ ]:
!python experiments/exp02_noisy_data.py \
    --phase1-epochs 30 --phase2-epochs 120 \
    --hidden-features 64 --hidden-layers 3 \
    --n-trials 2 \
    --noise-levels 0.0 0.05 0.10 0.15

## Full Noise Sensitivity Study

Use this after the smoke run succeeds. Runs 10 trials per noise level with full training.


In [ ]:
!python experiments/exp02_noisy_data.py \
    --phase1-epochs 500 --phase2-epochs 3000 \
    --n-trials 10 \
    --noise-levels 0.0 0.05 0.10 0.15

In [ ]:
!ls -lah /content/Vibration_PINN/results/noise_sensitivity

## Visualize Results

Load the saved `.npz` and plot error vs noise level.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path("/content/Vibration_PINN")
results_dir = PROJECT_DIR / "results" / "noise_sensitivity"
data = np.load(results_dir / "exp02_results.npz")

noise_levels = data["noise_levels"]
true_k = data["true_stiffnesses"]
true_xi = data["true_xi"]
n_floors = len(true_k)

# Compute relative errors per noise level
noise_pcts = [int(round(nl * 100)) for nl in noise_levels]
k_mean_err = []
k_std_err = []
xi_mean_err = []
xi_std_err = []

for pct in noise_pcts:
    label = f"{pct}pct"
    k_trials = data[f"stiffnesses_{label}"]  # (n_trials, n_floors)
    xi_trials = data[f"damping_{label}"]      # (n_trials,)

    k_rel = np.abs(k_trials - true_k) / true_k * 100  # (n_trials, n_floors)
    k_mean_err.append(k_rel.mean(axis=0))  # mean across trials per floor
    k_std_err.append(k_rel.std(axis=0))

    xi_rel = np.abs(xi_trials - true_xi) / true_xi * 100
    xi_mean_err.append(xi_rel.mean())
    xi_std_err.append(xi_rel.std())

k_mean_err = np.array(k_mean_err)
k_std_err = np.array(k_std_err)

# --- Plot stiffness errors ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x = np.array(noise_pcts)
for i in range(n_floors):
    ax.errorbar(x, k_mean_err[:, i], yerr=k_std_err[:, i],
                marker="o", capsize=4, label=f"k{i+1}")
ax.set_xlabel("Noise level (%)")
ax.set_ylabel("Relative error (%)")
ax.set_title("Stiffness identification error vs noise")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(x)

# --- Plot damping errors ---
ax = axes[1]
ax.errorbar(x, xi_mean_err, yerr=xi_std_err,
            marker="s", capsize=4, color="tab:red", label="xi")
ax.set_xlabel("Noise level (%)")
ax.set_ylabel("Relative error (%)")
ax.set_title("Damping identification error vs noise")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(x)

plt.tight_layout()
fig.savefig(results_dir / "exp02_noise_sensitivity.png", dpi=150)
plt.show()
print(f"Saved: {results_dir / 'exp02_noise_sensitivity.png'}")

In [ ]:
from pathlib import Path
from IPython.display import Image, display, Markdown
from google.colab import files

RESULTS_DIR = Path("/content/Vibration_PINN/results/noise_sensitivity")
result_files = list(RESULTS_DIR.glob("*.png")) + list(RESULTS_DIR.glob("*.npz"))

display(Markdown("### Generated files"))
for p in sorted(result_files):
    if p.suffix == ".png":
        display(Markdown(f"**{p.name}**"))
        display(Image(filename=str(p)))
    else:
        print(f"Data: {p.name}")

# Download files from Colab
for p in sorted(result_files):
    if p.exists():
        files.download(str(p))